In [1]:
%pip install torch ta mplfinance scikit-learn matplotlib pandas

import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import pandas as pd

# --- הגדרת המטבע ---
SYMBOL = 'BTCUSDT'
MODEL_TYPE = 'lstm'

# --- 1. חיבור לגוגל דרייב ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    print("Not running in Google Colab. Using local directory.")
    BASE_DIR = os.path.abspath(os.getcwd())

# Shared module (crypto_eval.py in the CryptoProject folder). Trainer only saves a bundle.
sys.path.append(BASE_DIR)
from crypto_eval import save_eval_bundle

# --- נתיבים ---
DATA_DIR = os.path.join(BASE_DIR, f'processed_data_{MODEL_TYPE}', SYMBOL)
MODEL_DIR = os.path.join(BASE_DIR, 'models')
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, f'best_{MODEL_TYPE}_model_{SYMBOL}.pth')
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# --- Hyperparameters ---
BATCH_SIZE = 128
EPOCHS = 50              # ceiling - EarlyStopping ends sooner
LEARNING_RATE = 0.001
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.2
EARLY_STOP_PATIENCE = 5


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        attn_weights = F.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context, attn_weights

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attention = Attention(hidden_size)
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size // 2, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        lstm_out, _ = self.lstm(x, (h0, c0))
        context, _ = self.attention(lstm_out)
        out = F.relu(self.fc1(context))
        out = self.dropout(out)
        out = self.fc2(out)
        # No sigmoid: Bollinger %B is not bounded to [0,1].
        return out


class EarlyStopping:
    def __init__(self, patience=5, delta=1e-6):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_loss = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'  EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        self.best_loss = val_loss


def load_data():
    print(f"Loading data from {DATA_DIR}...")
    X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
    y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
    X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
    y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    print(f"  Shapes -> X_train {X_train.shape} | y_test range [{y_test.min():.3f}, {y_test.max():.3f}] (should look like %B ~0..1)")
    return X_train, y_train, X_val, y_val, X_test, y_test


def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    X_train, y_train, X_val, y_val, X_test, y_test = load_data()

    # Features are ALREADY StandardScaled in preprocessing - do NOT scale again.
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_tensor, y_val_tensor),     batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test_tensor),                  batch_size=BATCH_SIZE, shuffle=False)

    N_features = X_train.shape[2]
    model = LSTMModel(input_size=N_features, hidden_size=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-5)
    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE, delta=1e-6)

    print(f"Starting training for {SYMBOL}...")
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                val_loss += criterion(outputs.squeeze(), y_batch).item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        scheduler.step(val_loss)
        print(f'Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # --- Predict on test set, then SAVE the results bundle (graphs are made in the eval notebook) ---
    print("\nEvaluating on Test Set...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    model.eval()
    predictions = []
    with torch.no_grad():
        for X_batch, in test_loader:
            predictions.extend(model(X_batch).squeeze().cpu().numpy())
    predictions = np.array(predictions).flatten()

    save_eval_bundle(BASE_DIR, SYMBOL, MODEL_TYPE, y_test, predictions, train_losses, val_losses)
    print(f"Done. Now run the eval notebook:  generate('{SYMBOL}', '{MODEL_TYPE}')")


if __name__ == "__main__":
    train()


Mounted at /content/drive
Google Drive mounted successfully!
Using device: cuda
Loading data from /content/drive/MyDrive/CryptoProject/processed_data_lstm/BTCUSDT...
  Shapes -> X_train (35573, 60, 16) | y_test range [-0.555, 1.501] (should look like %B ~0..1)
Starting training for BTCUSDT...
Epoch 1/50, Train Loss: 0.058957, Val Loss: 0.041957 | LR: 0.001000
Epoch 2/50, Train Loss: 0.038124, Val Loss: 0.027064 | LR: 0.001000
Epoch 3/50, Train Loss: 0.029220, Val Loss: 0.022840 | LR: 0.001000
Epoch 4/50, Train Loss: 0.024447, Val Loss: 0.020927 | LR: 0.001000
Epoch 5/50, Train Loss: 0.022711, Val Loss: 0.018647 | LR: 0.001000
Epoch 6/50, Train Loss: 0.021136, Val Loss: 0.016694 | LR: 0.001000
Epoch 7/50, Train Loss: 0.020218, Val Loss: 0.016155 | LR: 0.001000
Epoch 8/50, Train Loss: 0.019695, Val Loss: 0.017210 | LR: 0.001000
  EarlyStopping: 1/5
Epoch 9/50, Train Loss: 0.019202, Val Loss: 0.016439 | LR: 0.001000
  EarlyStopping: 2/5
Epoch 10/50, Train Loss: 0.018947, Val Loss: 0.01554